In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "POLUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,vol_regime_ratio,hour_sin,hour_cos,dow_sin,dow_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,0.2140,0.2142,0.2136,0.2137,95720.9,2025-06-01 00:04:59.999999+00:00,20471.83248,121,72999.9,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000e+00,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,0.2137,0.2138,0.2136,0.2138,24108.0,2025-06-01 00:09:59.999999+00:00,5151.14851,40,7831.3,...,NaN,0.0,1.0,-0.781831,0.62349,0.000008,1.595442e-06,0.000006,NaN,NaN
2,2025-06-01 00:10:00+00:00,0.2138,0.2140,0.2134,0.2136,124953.6,2025-06-01 00:14:59.999999+00:00,26696.71412,118,29028.5,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000002,9.127199e-07,-0.000003,NaN,NaN
3,2025-06-01 00:15:00+00:00,0.2136,0.2136,0.2132,0.2133,33999.9,2025-06-01 00:19:59.999999+00:00,7253.54456,73,9719.7,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000033,-5.950526e-06,-0.000027,NaN,NaN
4,2025-06-01 00:20:00+00:00,0.2133,0.2137,0.2132,0.2136,59750.9,2025-06-01 00:24:59.999999+00:00,12754.94294,111,42768.2,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000034,-1.152794e-05,-0.000022,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,433
[info] optuna train rows: 53,396
[info] valid rows:        13,350
[info] test rows:         16,687


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 16:17:42,440] A new study created in memory with name: no-name-9cda889a-7e84-4db7-b725-d6caf2009238


[I 2026-03-23 16:17:42,611] Trial 0 finished with value: 0.5258800624840588 and parameters: {'n_estimators': 500, 'learning_rate': 0.046187109390049115, 'max_depth': 5, 'subsample': 0.7996646210492592, 'colsample_bytree': 0.6890046601106091, 'colsample_bylevel': 0.6889986300840507, 'min_child_weight': 5, 'gamma': 2.5985284373248057, 'reg_alpha': 0.12306931514988033, 'reg_lambda': 8.341106432362084, 'scale_pos_weight': 0.9795721388475571}. Best is trial 0 with value: 0.5258800624840588.


[I 2026-03-23 16:17:42,855] Trial 1 finished with value: 0.5286834628291147 and parameters: {'n_estimators': 900, 'learning_rate': 0.03818145165896871, 'max_depth': 3, 'subsample': 0.6954562418017751, 'colsample_bytree': 0.6958511274633585, 'colsample_bylevel': 0.7260605607398845, 'min_child_weight': 13, 'gamma': 1.2958350559263474, 'reg_alpha': 0.010295300642650052, 'reg_lambda': 6.252287916406214, 'scale_pos_weight': 1.0138142002356034}. Best is trial 1 with value: 0.5286834628291147.


[I 2026-03-23 16:17:43,223] Trial 2 finished with value: 0.5267939120450633 and parameters: {'n_estimators': 500, 'learning_rate': 0.018033330377234345, 'max_depth': 4, 'subsample': 0.8462939903482534, 'colsample_bytree': 0.6999184455395899, 'colsample_bylevel': 0.778558609603403, 'min_child_weight': 14, 'gamma': 0.13935123815999317, 'reg_alpha': 0.12957079329680446, 'reg_lambda': 1.6666983286066417, 'scale_pos_weight': 0.9922397106016726}. Best is trial 1 with value: 0.5286834628291147.


[I 2026-03-23 16:17:43,396] Trial 3 finished with value: 0.5281054020029646 and parameters: {'n_estimators': 900, 'learning_rate': 0.047309442068985116, 'max_depth': 5, 'subsample': 0.7261534422933427, 'colsample_bytree': 0.6744180285015959, 'colsample_bylevel': 0.8210582566280392, 'min_child_weight': 12, 'gamma': 0.3661147045343365, 'reg_alpha': 0.05269751777340593, 'reg_lambda': 1.1085122517311703, 'scale_pos_weight': 1.2663797650885504}. Best is trial 1 with value: 0.5286834628291147.


...

[I 2026-03-23 16:18:07,572] Trial 96 finished with value: 0.5282164551330797 and parameters: {'n_estimators': 900, 'learning_rate': 0.046548021410882476, 'max_depth': 3, 'subsample': 0.6552800302105287, 'colsample_bytree': 0.6737169011877813, 'colsample_bylevel': 0.7831568563518944, 'min_child_weight': 7, 'gamma': 1.6899202431348583, 'reg_alpha': 0.05771932191484095, 'reg_lambda': 2.729471086181877, 'scale_pos_weight': 1.1536280725245092}. Best is trial 68 with value: 0.5329937032728523.


[I 2026-03-23 16:18:07,787] Trial 97 finished with value: 0.531032844214317 and parameters: {'n_estimators': 800, 'learning_rate': 0.03760139684523299, 'max_depth': 3, 'subsample': 0.673015561585168, 'colsample_bytree': 0.6863375951554418, 'colsample_bylevel': 0.7569190313135328, 'min_child_weight': 17, 'gamma': 1.0040773417669884, 'reg_alpha': 0.0834744293352789, 'reg_lambda': 2.2648647440877157, 'scale_pos_weight': 1.1847960383307183}. Best is trial 68 with value: 0.5329937032728523.


[I 2026-03-23 16:18:07,949] Trial 98 finished with value: 0.5264308142384806 and parameters: {'n_estimators': 800, 'learning_rate': 0.048528237870150724, 'max_depth': 3, 'subsample': 0.8626200191981575, 'colsample_bytree': 0.6838487934172907, 'colsample_bylevel': 0.7632480195353694, 'min_child_weight': 15, 'gamma': 1.2093013174154792, 'reg_alpha': 0.08550031365775415, 'reg_lambda': 3.015449102477136, 'scale_pos_weight': 1.185714354216138}. Best is trial 68 with value: 0.5329937032728523.


[I 2026-03-23 16:18:08,187] Trial 99 finished with value: 0.5300085283296975 and parameters: {'n_estimators': 800, 'learning_rate': 0.04135515664455777, 'max_depth': 3, 'subsample': 0.6748533492706167, 'colsample_bytree': 0.7151267781454985, 'colsample_bylevel': 0.7562676868400553, 'min_child_weight': 16, 'gamma': 1.027465658159846, 'reg_alpha': 0.012939942821653836, 'reg_lambda': 1.6052008371023476, 'scale_pos_weight': 1.2027595317313013}. Best is trial 68 with value: 0.5329937032728523.


['hour_sin', 'vol_30', 'dow_sin', 'hour_cos', 'mom_60', 'dist_ma_30', 'macd_hist', 'dow_cos', 'imbalance_15', 'atr_norm', 'vol_regime_ratio', 'dist_ma_15', 'trend_strength', 'vol_5', 'vol_ratio_5_30', 'mom_15', 'mom_5', 'range_ratio', 'volume_mom_5', 'imbalance_z', 'taker_buy_ratio', 'num_trades_mom_5', 'trades_z', 'volume_z', 'bar_range']
feature
hour_sin            10.045336
vol_30               9.930413
dow_sin              9.763704
hour_cos             9.381740
mom_60               9.127401
dist_ma_30           9.123650
macd_hist            9.011292
dow_cos              8.910207
imbalance_15         8.727752
atr_norm             8.690478
vol_regime_ratio     8.479401
dist_ma_15           8.249536
trend_strength       8.234131
vol_5                8.067863
vol_ratio_5_30       7.880552
mom_15               7.814892
mom_5                7.738382
range_ratio          7.694990
volume_mom_5         6.908021
imbalance_z          6.835803
taker_buy_ratio      6.740857
num_trades_mom_5    

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.207462
Test IC:         0.034724
Train ROC AUC:   0.620936
Test ROC AUC:    0.514249
Train PR AUC:    0.593344
Test PR AUC:     0.457553
Train Log Loss:  0.685168
Test Log Loss:   0.705981
Train Brier:     0.246088
Test Brier:      0.256333
Train Accuracy:  0.530339
Test Accuracy:   0.465332
Train Precision: 0.504136
Test Precision:  0.449356
Train Recall:    0.911545
Test Recall:     0.892299
Train F1:        0.649218
Test F1:         0.597709


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.358, 0.497] -0.000486   1669  0.006856
(0.497, 0.511] -0.000169   1669  0.006801
(0.511, 0.522]  0.000001   1668  0.007010
(0.522, 0.53]  -0.000051   1669  0.006693
(0.53, 0.538]  -0.000424   1669  0.006452
(0.538, 0.546]  0.000056   1668  0.006407
(0.546, 0.554] -0.000194   1669  0.007440
(0.554, 0.566]  0.000112   1668  0.006417
(0.566, 0.584]  0.000084   1669  0.006037
(0.584, 0.74]   0.000155   1669  0.009933


/tmp/ipykernel_1573869/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/POLUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/POLUSDT__h6_model.joblib
[saved] features -> models/xgb/POLUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/POLUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/POLUSDT__h6_meta.json
